### Data downloading

In [1]:
# !kaggle datasets download gowrishankarp/newspaper-text-summarization-cnn-dailymail -p .

In [2]:
# import zipfile
# with zipfile.ZipFile("newspaper-text-summarization-cnn-dailymail.zip", "r") as zip_ref:
#     zip_ref.extractall(".")  

In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import random
import matplotlib.pyplot as plt
import nltk
from torch import amp
import torch.nn.functional as F
import os
import json
from torch.utils.data import Dataset, DataLoader
import math

nltk.download('punkt_tab')


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [4]:
train_df=pd.read_csv('cnn_dailymail/train.csv')
val_df=pd.read_csv('cnn_dailymail/validation.csv')
test_df=pd.read_csv('cnn_dailymail/test.csv')
print(train_df.columns)

Index(['id', 'article', 'highlights'], dtype='object')


- Plot out the lengths of article to truncate the dataset because training resource is limited

In [5]:
# import matplotlib.pyplot as plt
# import seaborn as sns
# train_df['article_length'] = train_df['article'].apply(lambda x: len(x.split()))

# sns.set_theme(style="whitegrid")

# plt.figure(figsize=(10, 6))
# sns.histplot(train_df['article_length'], bins=50, kde=True)
# plt.title('Distribution of Article Lengths (in words)')
# plt.xlabel('Number of Words')
# plt.ylabel('Number of Articles')
# plt.show()


### Data preprocessing

In [5]:
from nltk.tokenize import word_tokenize

def tokenize(sentence):
    return word_tokenize(sentence.lower())



def build_vocab(sentences,  vocab_save_path="vocab.json", max_vocab_size=50000, min_freq=2):
    if os.path.exists(vocab_save_path):
        print("Loading vocab from file...")
        with open(vocab_save_path, 'r', encoding='utf-8') as f:
            vocab = json.load(f)
        print(f"Loaded vocab size: {len(vocab)}")
        return vocab

    print("Tokenizing sentences...")
    tokenized_sentences = [tokenize(sentence) for sentence in sentences]

    freq = {}
    for sent in tokenized_sentences:
        for word in sent:
            freq[word] = freq.get(word, 0) + 1

    sorted_words = sorted(freq.items(), key=lambda item: item[1], reverse=True)

    vocab = {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3}
    for word, count in sorted_words:
        if count < min_freq:
            continue
        if len(vocab) >= max_vocab_size:
            break
        vocab[word] = len(vocab)

    print(f"Final vocab size: {len(vocab)}")

    with open(vocab_save_path, 'w', encoding='utf-8') as f:
        json.dump(vocab, f, ensure_ascii=False)

    return vocab


# Example usage
vocab = build_vocab(train_df['article'],
                        vocab_save_path="vocab.json")

print(list(vocab)[:10])


Loading vocab from file...
Loaded vocab size: 50000
['<pad>', '<sos>', '<eos>', '<unk>', 'the', '.', ',', 'to', 'a', 'and']


In [6]:
def encode(sentence, vocab):
    return [vocab.get(w, vocab['<unk>']) for w in tokenize(sentence)]

### Dataset and DataLoader prepared

In [7]:
class SummarizationDataset(torch.utils.data.Dataset):
    def __init__(self, src_sentences, tgt_sentences, vocab, max_len=750):
        self.data = []
        for src, tgt in zip(src_sentences, tgt_sentences):
            src_ids = encode(src, vocab)
            tgt_ids = [vocab['<sos>']] + encode(tgt, vocab) + [vocab['<eos>']]
            
            if len(src_ids) == 0 or len(tgt_ids) == 0:
                continue  

            if len(src_ids) < max_len:
                self.data.append((src_ids, tgt_ids))

        self.vocab = vocab

    def __getitem__(self, idx):
        src, tgt = self.data[idx]
        return src, tgt 

    def __len__(self):
        return len(self.data)


def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)

    src_lengths = torch.tensor([len(s) for s in src_batch])
    tgt_lengths = torch.tensor([len(t) for t in tgt_batch])

    src_pad = torch.nn.utils.rnn.pad_sequence([torch.tensor(s) for s in src_batch],
                                              batch_first=True, padding_value=0)
    tgt_pad = torch.nn.utils.rnn.pad_sequence([torch.tensor(t) for t in tgt_batch],
                                              batch_first=True, padding_value=0)
    return src_pad, src_lengths, tgt_pad, tgt_lengths



if os.path.exists('train_dataset_HEX2.pt') and os.path.exists('valid_dataset_HEX2.pt'):
    train_dataset = torch.load('train_dataset_HEX2.pt')
    valid_dataset = torch.load('valid_dataset_HEX2.pt')
else:
    train_dataset = SummarizationDataset(train_df['article'], train_df['highlights'], vocab)
    valid_dataset = SummarizationDataset(val_df['article'], val_df['highlights'], vocab)
    torch.save(train_dataset, 'train_dataset_HEX2.pt')
    torch.save(valid_dataset, 'valid_dataset_HEX2.pt')


train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn, pin_memory=True
)

valid_loader = torch.utils.data.DataLoader(
    valid_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn, pin_memory=True
)




C:\Users\Admin\AppData\Local\Temp\ipykernel_2044\1784762820.py:39: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_dataset = torch.load('train_dataset_HEX2.pt')
C:\Users

In [8]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers=2, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hid_dim, num_layers=n_layers, bidirectional=False, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src,src_lengths):
        embedded = self.dropout(self.embedding(src))  # [B, L, E]
        packed_embedded = nn.utils.rnn.pack_padded_sequence(embedded, src_lengths.cpu(), batch_first=True, enforce_sorted=False)

        packed_outputs, hidden = self.rnn(packed_embedded)         # outputs: [B, L, 2H], hidden: [2, B, H]
        outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_outputs, batch_first=True)  
        # outputs: [B, L, 2H], hidden: [2, B, H], we need to combine forward and backward hidden states
        hidden = hidden[-1]  # [B, H]
        return outputs, hidden  # outputs: encoder outputs, hidden: decoder initial hidden 

class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.scale = math.sqrt(hid_dim)

    def forward(self, hidden, encoder_outputs, src_mask):
        # hidden: [batch_size, hidden_dim]   (from Decoder, at current timestep)
        # encoder_outputs: [batch_size, src_len, hidden_dim] (from Encoder)

        # reshape hidden to [batch_size, 1, hidden_dim] to perform batch matmul
        hidden = hidden.unsqueeze(1)  # [B, 1, H]
        # [B, 1, H] × [B, H, src_len] -> [B, 1, src_len]
        energy = torch.bmm(hidden, encoder_outputs.transpose(1, 2)).squeeze(1)  # [B, src_len]
        energy = energy / self.scale
        energy = energy.masked_fill(src_mask == 0, -1e4)

        # apply softmax to get attention weights
        attention = torch.softmax(energy, dim=1)  # [B, src_len]

        return attention

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hid_dim, dec_hid_dim, attention, dropout=0.5):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(enc_hid_dim + emb_dim, dec_hid_dim, batch_first=True)
        self.fc_out = nn.Linear(dec_hid_dim + dec_hid_dim + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, encoder_outputs, src_mask):
        # input: [B], hidden: [B, dec_hid_dim], encoder_outputs: [B, src_len, enc_hid_dim]

        input = input.unsqueeze(1)  
        embedded = self.dropout(self.embedding(input))  

        attn_weights = self.attention(hidden, encoder_outputs,src_mask) 

        attn_weights = attn_weights.unsqueeze(1)  

        context = torch.bmm(attn_weights, encoder_outputs)  
        context = self.dropout(context)  
        rnn_input = torch.cat((embedded, context), dim=2)  
        
        hidden = hidden.unsqueeze(0)  
        output, hidden = self.rnn(rnn_input, hidden)  

        output = output.squeeze(1)  
        context = context.squeeze(1)  
        embedded = embedded.squeeze(1)  
        output = self.dropout(output)  
        prediction = self.fc_out(torch.cat((output, context, embedded), dim=1))  

        attn_weights = attn_weights.squeeze(1) 
        return prediction, hidden.squeeze(0), attn_weights 

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src,src_lengths, tgt, teacher_forcing_ratio=0.5):
        B, T = tgt.shape
        outputs = torch.zeros(B, T, self.decoder.output_dim).to(self.device,non_blocking=True)
        src_mask = (src != 0).to(self.device,non_blocking=True)
        encoder_outputs, hidden = self.encoder(src,src_lengths)

        input = tgt[:, 0]
        for t in range(1, T):
            output, hidden,_ = self.decoder(input, hidden, encoder_outputs, src_mask)
            outputs[:, t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = tgt[:, t] if teacher_force else top1

        return outputs


In [9]:
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
INPUT_DIM = len(vocab)
OUTPUT_DIM = len(vocab)
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 256


enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM)
attn = Attention(HID_DIM)  
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, HID_DIM, attn) 

model = Seq2Seq(enc, dec, device).to(device,non_blocking=True)

if os.path.exists('best-model-HEX2.pt'):
    model.load_state_dict(torch.load('best-model-HEX2.pt'))
    print("Loaded the best model from checkpoint")
else:
    print("No checkpoint found, starting training from scratch")

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding


cuda


C:\Users\Admin\AppData\Local\Temp\ipykernel_2044\3484235004.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best-model-HEX2.pt'))


Loaded the best model from checkpoint


### Training loop:

In [10]:
from torch import amp

scaler = amp.GradScaler('cuda')

def train(model, iterator, optimizer, criterion, teacher_forcing_ratio, clip=1,grad_accum_steps=1):
    model.train()
    epoch_loss = 0

    progress_bar = tqdm(iterator, desc="Training", leave=False)
    optimizer.zero_grad()

    for i, (src, src_lengths, tgt, tgt_lengths) in enumerate(progress_bar):
        src, tgt = src.to(device,non_blocking=True), tgt.to(device,non_blocking=True)

        with amp.autocast('cuda'):
            output = model(src, src_lengths, tgt, teacher_forcing_ratio=teacher_forcing_ratio)  # [B, T, vocab_size]
            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            tgt = tgt[:, 1:].reshape(-1)
            loss = criterion(output, tgt)
            loss = loss / grad_accum_steps  

        scaler.scale(loss).backward()

        if (i + 1) % grad_accum_steps == 0 or (i + 1) == len(iterator):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        batch_loss = loss.item() * grad_accum_steps  
        epoch_loss += batch_loss

        progress_bar.set_postfix(
            batch_loss=f"{batch_loss:.4f}",
            avg_loss=f"{epoch_loss / (i + 1):.4f}"
        )

    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src, src_length, tgt,tgt_length in tqdm(iterator, desc="Evaluating", leave=False):
            src, tgt = src.to(device,non_blocking=True), tgt.to(device,non_blocking=True)
            with amp.autocast('cuda'):    
                output = model(src,src_length, tgt, teacher_forcing_ratio=0)  
                output_dim = output.shape[-1]
                output = output[:, 1:].reshape(-1, output_dim)
                tgt = tgt[:, 1:].reshape(-1)
                loss = criterion(output, tgt)

            epoch_loss += loss.item()

    return epoch_loss / len(iterator)


In [ ]:
N_EPOCHS = 10
CLIP = 1
patience = 3
best_valid_loss = float('inf')
counter = 0

for epoch in range(N_EPOCHS):
    teacher_forcing_ratio = 0.5
    train_loss = train(model, train_loader, optimizer, criterion, teacher_forcing_ratio, CLIP,grad_accum_steps=4)
    valid_loss = evaluate(model, valid_loader, criterion)

    print(f'Epoch {epoch+1}:')
    print(f'  Train Loss = {train_loss:.3f}')
    print(f'  Valid Loss = {valid_loss:.3f}')

    # Save the best model
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        counter = 0
        torch.save(model.state_dict(), 'best-model-HEX2.pt')
        print('  Validation loss improved, model saved!')
    else:
        counter += 1
        print(f'  No improvement ({counter}/{patience})')

    # Early stopping
    if counter >= patience:
        print('Early stopping triggered!')
        break


Epoch 1:
  Train Loss = 6.098
  Valid Loss = 6.620
  Validation loss improved, model saved!


Training:   0%|          | 1/38357 [00:00<3:43:47,  2.86it/s, avg_loss=6.1174, batch_loss=6.1174]

### Summarize using beam search and calculate ROUGE scores

In [11]:
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F
from collections import defaultdict


def summarize(model, text, vocab, beam_width=10, max_len=150, length_penalty_alpha=0.7):
    model.eval()
    tokens = tokenize(text)
    src_ids = [vocab.get(tok, vocab['<unk>']) for tok in tokens]
    src_len = len(src_ids)

    src_tensor = pad_sequence([torch.tensor(src_ids)], batch_first=True, padding_value=vocab['<pad>']).to(device, non_blocking=True)
    src_lengths = torch.tensor([src_len], device=device)
    src_mask = (torch.tensor(src_ids) != vocab['<pad>']).to(device, non_blocking=True)

    with torch.no_grad():
        with amp.autocast('cuda'):
            encoder_outputs, hidden = model.encoder(src_tensor, src_lengths)

    inv_vocab = {v: k for k, v in vocab.items()}
    beam = [([vocab['<sos>']], hidden, 0.0, set())]  # (sequence, hidden, score, trigrams)

    for _ in range(max_len):
        new_beam = []
        for seq, hidden, score, trigrams in beam:
            if seq[-1] == vocab['<eos>']:
                new_beam.append((seq, hidden, score, trigrams))
                continue

            tgt_tensor = torch.tensor([seq[-1]]).to(device, non_blocking=True)
            with torch.no_grad():
                with amp.autocast('cuda'):
                    output, hidden, _ = model.decoder(tgt_tensor, hidden, encoder_outputs, src_mask)
                    log_probs = F.log_softmax(output, dim=1)
                    topk_scores, topk_tokens = torch.topk(log_probs, beam_width, dim=1)

            for i in range(beam_width):
                token = topk_tokens[0][i].item()
                if token==vocab['<unk>']:
                    continue
                token_score = topk_scores[0][i].item()
                new_seq = seq + [token]

                # Repetition penalty (consecutive tokens)
                repetition_penalty = 5.0 if token == seq[-1] else 0.0

                # Trigram diversity penalty
                trigram_penalty = 0.0
                if len(seq) >= 2:
                    trigram = (seq[-2], seq[-1], token)
                    if trigram in trigrams:
                        trigram_penalty = 5.0
                new_trigrams = set(trigrams)
                if len(new_seq) >= 3:
                    new_trigrams.add(tuple(new_seq[-3:]))

                # Length penalty
                length_penalty = ((5 + len(new_seq)) / 6) ** length_penalty_alpha
                adjusted_score = (score + token_score - repetition_penalty - trigram_penalty) / length_penalty

                new_beam.append((new_seq, hidden, adjusted_score, new_trigrams))

        # Prune top beam_width
        beam = sorted(new_beam, key=lambda x: x[2], reverse=True)[:beam_width]

        # Early stop if all beams ended with <eos>
        if all(seq[-1] == vocab['<eos>'] for seq, _, _, _ in beam):
            break

    # Select best sequence
    best_seq, _, _, _ = max(beam, key=lambda x: x[2])

    # Convert to words and filter out <unk>
    inv_vocab = {v: k for k, v in vocab.items()}
    summary = [inv_vocab.get(idx, '<unk>') for idx in best_seq[1:] if inv_vocab.get(idx, '<unk>') != '<unk>' and idx != vocab['<eos>']]

    return summary


In [12]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

test = test_df['article'][2000]
reference=test_df['highlights'][2000]
summary = summarize(model, test, vocab)
summary=" ".join(summary)
print("Test: ", test)
print("-"*100)
print("Reference: ", reference)
print("-"*100)
print("Prediction: ", summary)
print("-"*100)
scores = scorer.score(reference, summary)
for metric, score in scores.items():
    print(f"{metric}:  F1: {score.fmeasure:.4f}")



Test:  Don't be fooled, Arsene. Don’t be caught out by this impressive run of form and think everything is in place for a crack at the title next year. It is not uncommon to see Arsenal finish a season with a string of good results. Usually they rattle out a sequence of wins just in time to preserve their place in the top four but too late to affect the destiny of the Barclays Premier League. The same thing has happened this year. Chelsea were relentless from the first whistle and have had the title race firmly under control since before Christmas. Arsenal players celebrate after their extra-time winner against Reading in the FA Cup semi-finals last week . Alexis Sanchez jumps in the air and celebrates wildly after his second goal for Arsenal at Wembley . With a 10-point advantage, even if Jose Mourinho loses to Arsene Wenger for the first time on Sunday it won’t stop his side becoming champions. Next year, though, Chelsea won’t have it all their own way and no team is better placed th

In [ ]:
from tqdm import tqdm

sample_size = len(test_df)  # integer division for 10%
total_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}

for idx in tqdm(range(sample_size), desc="Scoring summaries "):
    article = test_df['article'][idx]
    reference = test_df['highlights'][idx]
    summary = summarize(model, article, vocab, beam_width=5, max_len=150)
    summary = " ".join(summary)
    
    scores = scorer.score(reference, summary)
    for metric in total_scores:
        total_scores[metric].append(scores[metric].fmeasure)

avg_scores = {metric: np.mean(total_scores[metric]) for metric in total_scores}
print("\nAverage ROUGE scores over sample:")
for metric, avg in avg_scores.items():
    print(f"{metric}: F1: {avg:.4f}")


Scoring summaries :   1%|          | 94/11490 [01:01<2:41:14,  1.18it/s]